# Notebook 03: Model Training

**Project:** GNN-Based Battery Voltage Predictor  

---

## Overview

This notebook trains three models for Li intercalation voltage prediction:

1. **Random Forest (RF)**: Scikit-learn RF on matminer compositional descriptors.
   Fast, interpretable baseline. No GPU required.

2. **CGCNN (from scratch)**: Crystal Graph Convolutional Neural Network implemented
   with PyTorch Geometric. Four convolutional layers, global mean pooling,
   two-layer MLP head. Trained with Adam optimizer, ReduceLROnPlateau scheduler,
   and early stopping. Target: MAE less than 0.3 V.

3. **M3GNet (fine-tuned)**: Pretrained M3GNet backbone from matgl, with the
   regression head replaced and fine-tuned on the battery dataset. Two-phase
   training: (a) warm-up with backbone frozen, (b) joint fine-tuning at lower LR.

All models are saved to `models/`. Loss curves and training configs are also saved.

In [ ]:
import sys
import json
import pickle
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import VoltageGraphDataset
from src.models import build_cgcnn, M3GNetVoltagePredictor
from src.train import train_cgcnn, make_loaders, predict
from src.evaluate import plot_loss_curves, PALETTE
from src.utils import set_seed

set_seed(42)

DATA_DIR   = project_root / 'data'
MODELS_DIR = project_root / 'models'
RESULTS_DIR = project_root / 'results'
MODELS_DIR.mkdir(exist_ok=True)

# Detect GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Load PyG graph datasets
train_ds = VoltageGraphDataset.from_processed(str(DATA_DIR / 'train_graphs.pkl'))
val_ds   = VoltageGraphDataset.from_processed(str(DATA_DIR / 'val_graphs.pkl'))
test_ds  = VoltageGraphDataset.from_processed(str(DATA_DIR / 'test_graphs.pkl'))

print(f'Graph datasets: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')

# Load matminer features
with open(DATA_DIR / 'matminer_filtered.pkl', 'rb') as f:
    mm = pickle.load(f)

X_train, y_train = mm['X_train'], mm['y_train']
X_val,   y_val   = mm['X_val'],   mm['y_val']
X_test,  y_test  = mm['X_test'],  mm['y_test']
feat_names = mm['feature_names']

print(f'Matminer features: {X_train.shape}')

## 1. Random Forest Baseline

We fit a Random Forest on the matminer feature matrix. Hyperparameters are
selected via a small random search over n_estimators (100 to 500) and
max_features ('sqrt', 0.3, 0.5). The best model is evaluated on the validation
set and the final test set performance is held out until Notebook 04.

Random forests provide:
- A strong non-parametric baseline without GPU requirements
- Feature importances for chemical interpretation
- Prediction intervals via the variance of individual tree predictions

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error

# Combine train+val for hyperparameter search
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

# Quick random search
param_dist = {
    'n_estimators': [200, 300, 400, 500],
    'max_features': ['sqrt', 0.3, 0.5],
    'min_samples_leaf': [1, 2, 4],
    'max_depth': [None, 20, 30],
}

rf_base = RandomForestRegressor(n_jobs=-1, random_state=42)
rsearch = RandomizedSearchCV(
    rf_base, param_dist, n_iter=12, cv=3,
    scoring='neg_mean_absolute_error', random_state=42, verbose=1, n_jobs=-1
)

print('Running random search (12 iterations, 3-fold CV)...')
t0 = time.time()
rsearch.fit(X_trainval, y_trainval)
print(f'Done in {time.time() - t0:.1f}s')
print(f'Best params: {rsearch.best_params_}')
print(f'Best CV MAE: {-rsearch.best_score_:.4f} V')

rf_model = rsearch.best_estimator_

In [ ]:
# Evaluate on validation set
rf_val_pred = rf_model.predict(X_val)
rf_val_mae = mean_absolute_error(y_val, rf_val_pred)
print(f'RF Validation MAE: {rf_val_mae:.4f} V')

# Save model
with open(MODELS_DIR / 'rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Save config
with open(MODELS_DIR / 'rf_config.json', 'w') as f:
    json.dump({
        'best_params': rsearch.best_params_,
        'cv_mae': float(-rsearch.best_score_),
        'val_mae': float(rf_val_mae),
        'n_train': len(X_train),
        'n_features': X_train.shape[1],
    }, f, indent=2)

print(f'Saved RF model to {MODELS_DIR / "rf_model.pkl"}')

## 2. CGCNN From Scratch

Architecture summary:
- Node embedding: Linear(9, 128)
- 4 CGConv layers (edge_dim=64) with BatchNorm, ReLU, residual skip connections
- Global mean pooling
- MLP head: Linear(128, 64) -> Softplus -> Dropout(0.1) -> Linear(64, 1)

Training setup:
- Loss: L1 (MAE)
- Optimizer: Adam, lr=1e-3, weight_decay=1e-5
- Scheduler: ReduceLROnPlateau (patience=10, factor=0.5)
- Early stopping: patience=25 epochs
- Batch size: 64
- Max epochs: 300

In [ ]:
# Build CGCNN
cgcnn = build_cgcnn(
    node_dim=9, edge_dim=64,
    hidden_dim=128, n_conv=4, dropout=0.1
)
print(f'CGCNN parameters: {cgcnn.count_parameters():,}')
print(cgcnn)

In [ ]:
train_loader, val_loader, test_loader = make_loaders(
    train_ds, val_ds, test_ds,
    batch_size=64, num_workers=4
)

print('Starting CGCNN training...')
cgcnn_history = train_cgcnn(
    model=cgcnn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    n_epochs=300,
    lr=1e-3,
    weight_decay=1e-5,
    patience=25,
    lr_patience=10,
    lr_factor=0.5,
    save_dir=str(MODELS_DIR),
    model_name='cgcnn',
    verbose=True,
)
print('CGCNN training complete.')

In [ ]:
fig = plot_loss_curves(
    cgcnn_history, model_name='CGCNN',
    save_path=str(RESULTS_DIR / 'fig03_cgcnn_loss_curves.png')
)
plt.show()

best_val = min(cgcnn_history['val_loss'])
print(f'Best CGCNN val MAE: {best_val:.4f} V (epoch {cgcnn_history["best_epoch"]})')

## 3. M3GNet Fine-Tuning

M3GNet is a graph network potential pretrained on the Materials Project PES dataset
using a many-body 3-body interaction framework (matgl library).

Fine-tuning strategy:
- Phase 1 (warm-up, 20 epochs): freeze backbone, train only the new regression head
  at lr=1e-3. This prevents early-epoch gradient explosion in the pretrained layers.
- Phase 2 (joint, up to 200 more epochs): unfreeze all layers, train with lr=1e-4
  for the backbone and lr=1e-3 for the head (differential learning rates via
  parameter groups). Early stopping with patience=20 on validation MAE.

Note: matgl's M3GNet uses DGL as its graph backend, so structures are featurized
using matgl's internal Structure2Graph before passing to the model.

In [ ]:
import matgl
from matgl.ext.pymatgen import Structure2Graph, get_element_list
from matgl.utils.training import ModelLightningModule
from pymatgen.core import Structure
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from src.train import EarlyStopping

print(f'matgl version: {matgl.__version__}')

# Load pretrained M3GNet
print('Loading pretrained M3GNet-MP-2021.2.8-PES...')
pretrained_m3g = matgl.load_model('M3GNet-MP-2021.2.8-PES')
print('Pretrained model loaded.')

In [ ]:
# Build matgl-compatible dataset
from torch.utils.data import DataLoader as TorchDataLoader
import dgl

class M3GNetVoltageDataset(torch.utils.data.Dataset):
    """
    Wraps battery entries for matgl graph featurization.
    """
    def __init__(self, entries, converter, struct_key='charged_structure'):
        self.graphs = []
        self.targets = []
        self.valid_indices = []

        for i, entry in enumerate(entries):
            struct_dict = entry.get(struct_key) or entry.get('discharged_structure')
            if struct_dict is None:
                continue
            try:
                structure = Structure.from_dict(struct_dict)
                element_types = get_element_list([structure])
                graph, lat, state = converter.get_graph(structure)
                self.graphs.append((graph, lat, state))
                self.targets.append(entry['average_voltage'])
            except Exception:
                continue

        print(f'M3GNet dataset: {len(self.graphs)} valid structures')

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        return self.graphs[idx], torch.tensor(self.targets[idx], dtype=torch.float32)

# Collect all unique elements
all_entries = train_entries + val_entries + test_entries
all_structs = []
for e in all_entries:
    sd = e.get('charged_structure') or e.get('discharged_structure')
    if sd:
        try:
            all_structs.append(Structure.from_dict(sd))
        except Exception:
            pass

element_types = get_element_list(all_structs)
print(f'Element types in dataset: {len(element_types)}')

converter = Structure2Graph(element_types=element_types, cutoff=5.0)

m3g_train_ds = M3GNetVoltageDataset(train_entries, converter)
m3g_val_ds   = M3GNetVoltageDataset(val_entries,   converter)
m3g_test_ds  = M3GNetVoltageDataset(test_entries,  converter)

In [ ]:
def m3gnet_collate_fn(batch):
    graphs, targets = zip(*batch)
    dgl_graphs = [g[0] for g in graphs]
    lattices   = [g[1] for g in graphs]
    states     = [g[2] for g in graphs]
    batched_graphs = dgl.batch(dgl_graphs)
    batched_lat    = torch.stack(lattices)
    batched_state  = torch.stack(states) if states[0] is not None else None
    targets_t = torch.stack(targets)
    return (batched_graphs, batched_lat, batched_state), targets_t

m3g_train_loader = TorchDataLoader(
    m3g_train_ds, batch_size=32, shuffle=True,
    collate_fn=m3gnet_collate_fn, num_workers=2
)
m3g_val_loader = TorchDataLoader(
    m3g_val_ds, batch_size=32, shuffle=False,
    collate_fn=m3gnet_collate_fn, num_workers=2
)

# Build M3GNet voltage predictor
class M3GNetRegressor(nn.Module):
    def __init__(self, backbone, hidden_dim=64):
        super().__init__()
        self.backbone = backbone
        # Determine backbone embedding dimension
        backbone_dim = getattr(backbone.model, 'dim_node_embedding', 64)
        self.head = nn.Sequential(
            nn.Linear(backbone_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, g, lat, state):
        # Get node embeddings from backbone
        node_feats = self.backbone.model.get_graph_features(g, lat, state)
        # Pool over nodes
        pooled = dgl.mean_nodes(g, feat='h') if 'h' in g.ndata else node_feats.mean(0)
        return self.head(pooled).squeeze(-1)

m3gnet_model = M3GNetRegressor(pretrained_m3g, hidden_dim=64).to(device)
print(f'M3GNet regressor ready on {device}')

In [ ]:
# Phase 1: warm-up (backbone frozen)
for param in m3gnet_model.backbone.parameters():
    param.requires_grad = False

optimizer_warmup = Adam(filter(lambda p: p.requires_grad, m3gnet_model.parameters()), lr=1e-3)
criterion = nn.L1Loss()

print('Phase 1: Warm-up (backbone frozen, 20 epochs)...')
m3g_history = {'train_loss': [], 'val_loss': [], 'lr': []}

for epoch in range(1, 21):
    m3gnet_model.train()
    epoch_loss = 0.0
    n_samples = 0
    for (g, lat, state), targets in m3g_train_loader:
        g = g.to(device)
        lat = lat.to(device)
        if state is not None:
            state = state.to(device)
        targets = targets.to(device)
        optimizer_warmup.zero_grad()
        pred = m3gnet_model(g, lat, state)
        loss = criterion(pred, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(m3gnet_model.parameters(), 10.0)
        optimizer_warmup.step()
        epoch_loss += loss.item() * len(targets)
        n_samples += len(targets)
    train_loss = epoch_loss / n_samples

    m3gnet_model.eval()
    val_loss_total = 0.0
    val_n = 0
    with torch.no_grad():
        for (g, lat, state), targets in m3g_val_loader:
            g = g.to(device)
            lat = lat.to(device)
            if state is not None:
                state = state.to(device)
            targets = targets.to(device)
            pred = m3gnet_model(g, lat, state)
            val_loss_total += criterion(pred, targets).item() * len(targets)
            val_n += len(targets)
    val_loss = val_loss_total / val_n

    m3g_history['train_loss'].append(train_loss)
    m3g_history['val_loss'].append(val_loss)
    m3g_history['lr'].append(optimizer_warmup.param_groups[0]['lr'])

    if epoch % 5 == 0:
        print(f'Warm-up epoch {epoch:2d}: train MAE {train_loss:.4f} V, val MAE {val_loss:.4f} V')

print('Phase 1 complete.')

In [ ]:
# Phase 2: joint fine-tuning (all layers)
for param in m3gnet_model.backbone.parameters():
    param.requires_grad = True

optimizer_ft = Adam([
    {'params': m3gnet_model.backbone.parameters(), 'lr': 5e-5},
    {'params': m3gnet_model.head.parameters(),     'lr': 5e-4},
], weight_decay=1e-5)
scheduler_ft = ReduceLROnPlateau(optimizer_ft, mode='min', factor=0.5, patience=8, verbose=True)
early_stop = EarlyStopping(patience=20)

best_m3g_state = None

print('Phase 2: Joint fine-tuning (backbone + head)...')
for epoch in range(1, 201):
    m3gnet_model.train()
    epoch_loss = 0.0
    n_samples = 0
    for (g, lat, state), targets in m3g_train_loader:
        g = g.to(device)
        lat = lat.to(device)
        if state is not None:
            state = state.to(device)
        targets = targets.to(device)
        optimizer_ft.zero_grad()
        pred = m3gnet_model(g, lat, state)
        loss = criterion(pred, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(m3gnet_model.parameters(), 10.0)
        optimizer_ft.step()
        epoch_loss += loss.item() * len(targets)
        n_samples += len(targets)
    train_loss = epoch_loss / n_samples

    m3gnet_model.eval()
    val_loss_total = 0.0
    val_n = 0
    with torch.no_grad():
        for (g, lat, state), targets in m3g_val_loader:
            g = g.to(device)
            lat = lat.to(device)
            if state is not None:
                state = state.to(device)
            targets = targets.to(device)
            pred = m3gnet_model(g, lat, state)
            val_loss_total += criterion(pred, targets).item() * len(targets)
            val_n += len(targets)
    val_loss = val_loss_total / val_n

    scheduler_ft.step(val_loss)
    m3g_history['train_loss'].append(train_loss)
    m3g_history['val_loss'].append(val_loss)
    m3g_history['lr'].append(optimizer_ft.param_groups[1]['lr'])

    if val_loss < early_stop.best_loss:
        best_m3g_state = {k: v.clone() for k, v in m3gnet_model.state_dict().items()}

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}: train MAE {train_loss:.4f} V, val MAE {val_loss:.4f} V')

    if early_stop(val_loss, epoch):
        print(f'Early stopping at epoch {epoch}. Best val MAE: {early_stop.best_loss:.4f} V')
        break

m3gnet_model.load_state_dict(best_m3g_state)
print('Phase 2 complete.')

In [ ]:
torch.save(m3gnet_model.state_dict(), MODELS_DIR / 'm3gnet_best.pt')
with open(MODELS_DIR / 'm3gnet_history.json', 'w') as f:
    json.dump(m3g_history, f, indent=2)
print(f'Saved M3GNet model and history.')

fig = plot_loss_curves(
    m3g_history, model_name='M3GNet',
    save_path=str(RESULTS_DIR / 'fig03_m3gnet_loss_curves.png')
)
plt.show()

In [ ]:
print('Training complete. Model summary:')
print(f'  RF:     saved to models/rf_model.pkl')
print(f'  CGCNN:  saved to models/cgcnn_best.pt  (val MAE: {min(cgcnn_history["val_loss"]):.4f} V)')
print(f'  M3GNet: saved to models/m3gnet_best.pt (val MAE: {min(m3g_history["val_loss"]):.4f} V)')
print(f'\nProceed to 04_evaluation.ipynb')